# 01 — Classify a complaint

Vertical slice, now with retrieval. Cells 1-6 validated 2026-07-22 (clean
run). Cells 7-11 validated live the same day: upload, attach, and filtered
search all confirmed working end to end, including pinning down the correct
filter syntax (OpenAI's typed-filter convention, `{"type": "eq", "key":
..., "value": ...}`, not a flat dict, which silently returns zero results
rather than erroring). Cells 12-14 (population and retrieval) are new this
pass, following the same validated shapes, but not yet exercised against
the full 217-document corpus, only a single document. Treat Cell 12's
completion as the next real checkpoint.

**Run this after:**
- The workbench is running with the taxonomy ConfigMap mounted
  (manifests/workbench/workbench.yaml)
- `ansible-playbook ansible/site.yml` has completed (fully automated: seeds
  data, restarts the predictor once weights are confirmed complete, restarts
  Llama Stack so it discovers the model)

## Environment variables required

| Variable | Description | Default |
|---|---|---|
| `LLAMA_STACK_URL` | Llama Stack service | `http://lsd-complaint-intelligence-service:8321` |
| `GUARDRAILS_URL` | Guardrails orchestrator, HTTPS, self-signed | `https://guardrails-orchestrator-service:8032` |
| `MINIO_ENDPOINT` | MinIO service | `minio.complaint-intelligence.svc.cluster.local:9000` |
| `MINIO_ACCESS_KEY` | MinIO access key | none, required |
| `MINIO_SECRET_KEY` | MinIO secret key | none, required |
| `TAXONOMY_PATH` | Mounted ConfigMap path | `/opt/app-root/taxonomy/taxonomy.yaml` |

## Retrieval design (ADR-0008 addendum, 2026-07-22)

One vector store, not two. Each document carries a `kind` attribute
(`taxonomy` or `complaint`). At classification time, two filtered searches
run against the same store (one per kind, top-k 3 each), combined into the
prompt as two labeled sections. See the ADR-0008 addendum for full
reasoning.

## Cell 1 — Install dependencies

In [ ]:
import sys
!{sys.executable} -m pip install --quiet requests==2.32.3 pyyaml==6.0.2 minio==7.2.7
print("Dependencies installed.")
# Note: pip may warn about a urllib3 version conflict with appengine-python-standard,
# a package baked into the base workbench image and unrelated to this notebook.
# That warning is expected and harmless; only a raised exception here is a real failure.

## Cell 2 — Configuration

In [ ]:
import os

LLAMA_STACK_URL  = os.environ.get("LLAMA_STACK_URL", "http://lsd-complaint-intelligence-service:8321")
GUARDRAILS_URL   = os.environ.get("GUARDRAILS_URL", "https://guardrails-orchestrator-service:8032")
TAXONOMY_PATH    = os.environ.get("TAXONOMY_PATH", "/opt/app-root/taxonomy/taxonomy.yaml")

MINIO_ENDPOINT   = os.environ.get("MINIO_ENDPOINT", "minio.complaint-intelligence.svc.cluster.local:9000")
MINIO_ACCESS_KEY = os.environ["MINIO_ACCESS_KEY"]   # fail fast if not set
MINIO_SECRET_KEY = os.environ["MINIO_SECRET_KEY"]   # fail fast if not set

# NB: these are real bucket names, not the `mc` alias (`uc02`) used when
# seeding from a laptop. The alias is a local mc config convenience and is
# not part of the bucket name itself.
COMPLAINTS_BUCKET = "complaints"
EVIDENCE_BUCKET   = "evidence"
COMPLAINTS_KEY    = "incoming/records.jsonl"
EVIDENCE_PREFIX   = "classifications"

print("Configuration loaded.")

## Cell 3 — Load taxonomy from the mounted ConfigMap

In [ ]:
import yaml
from pathlib import Path

taxonomy_file = Path(TAXONOMY_PATH)
if not taxonomy_file.exists():
    raise RuntimeError(
        f"Taxonomy not found at {TAXONOMY_PATH}. Confirm the taxonomy "
        f"ConfigMap volume is mounted (manifests/workbench/workbench.yaml)."
    )

taxonomy = yaml.safe_load(taxonomy_file.read_text())
TAXONOMY_VERSION = taxonomy["version"]
CONFIDENCE_THRESHOLD = taxonomy["classification"]["confidence_threshold"]

print(f"Taxonomy version {TAXONOMY_VERSION}: "
      f"{len(taxonomy['themes'])} themes, {len(taxonomy['root_causes'])} root causes")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")

## Cell 4 — Discover the LLM at runtime

In [ ]:
import requests

resp = requests.get(f"{LLAMA_STACK_URL}/v1/models")
resp.raise_for_status()
models = resp.json().get("data", resp.json())

llm_candidates = [m for m in models if "granite" in m.get("id", "").lower()]
if not llm_candidates:
    raise RuntimeError(
        f"No Granite model found in /v1/models. Got: {models}\n"
        f"If Granite's InferenceService is healthy but still missing here, "
        f"restart the lsd-complaint-intelligence pod and re-run this cell "
        f"(ansible/site.yml now does this automatically after seeding)."
    )

MODEL_ID = llm_candidates[0]["id"]
print(f"Using model: {MODEL_ID}")

## Cell 5 — Pull one complaint from MinIO

In [ ]:
from minio import Minio
import json

minio_client = Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY,
                      secret_key=MINIO_SECRET_KEY, secure=False)

if not minio_client.bucket_exists(COMPLAINTS_BUCKET):
    raise RuntimeError(f"Bucket '{COMPLAINTS_BUCKET}' not found. Run ansible/site.yml.")

response = minio_client.get_object(COMPLAINTS_BUCKET, COMPLAINTS_KEY)
first_line = response.read().decode("utf-8").splitlines()[0]
response.close()
response.release_conn()

complaint = json.loads(first_line)
print(f"Loaded complaint {complaint['complaint_id']}")
print(complaint["body"][:300])

## Cell 6 — Guardrails input check

In [ ]:
detection_payload = {
    "detectors": {"regex": {"regex": ["email", "credit-card"]}},
    "content": complaint["body"]
}

# TLS is self-signed in-cluster (validated finding, DEPLOYMENT-LOG-2026-07-15).
# The InsecureRequestWarning this raises is expected and does not indicate failure.
detection_resp = requests.post(
    f"{GUARDRAILS_URL}/api/v2/text/detection/content",
    json=detection_payload,
    verify=False
)
detection_resp.raise_for_status()
detections = detection_resp.json()

pii_detected = len(detections) > 0
pii_redactions = len(detections)

# Note: only email/credit-card regex detectors are configured
# (manifests/guardrails/orchestrator-config.yaml). No injection detector is
# currently registered, so injection_blocked is not yet a live check, it is
# recorded honestly as not evaluated rather than assumed false.
injection_blocked = None

print(f"PII detected: {pii_detected} ({pii_redactions} matches)")
print(f"Injection check: not configured on this stack")

## Cell 7 — Retrieval route discovery

Confirms `/v1/vector_stores` and `/v1/files` are both present, needed
together since population attaches previously-uploaded file objects rather
than accepting raw text directly.

In [ ]:
resp = requests.get(f"{LLAMA_STACK_URL}/v1/inspect/routes")
resp.raise_for_status()
routes = resp.json()
all_routes = routes.get("data", routes)

vector_routes = [r for r in all_routes if "vector" in str(r).lower()]
file_routes = [r for r in all_routes if r.get("route", "").startswith("/v1/files")]

if not vector_routes:
    raise RuntimeError("No vector routes found. Confirm ENABLE_INLINE_MILVUS (ADR-0008).")
if not file_routes:
    raise RuntimeError(
        "No /v1/files routes found. Vector store population needs a file "
        "upload step first (OpenAI Vector Stores convention)."
    )

print("Vector/retrieval routes:")
for r in vector_routes:
    print(" ", r)
print("\nFile routes:")
for r in file_routes:
    print(" ", r)

## Cell 8 — Discover the embedding model

Per ADR-0008: read at runtime, never hardcoded. The embedding model
changed between RHOAI 2.25 and 3.4, so treating it as a constant would
guarantee breakage on the next platform version.

In [ ]:
resp = requests.get(f"{LLAMA_STACK_URL}/v1/models")
resp.raise_for_status()
models = resp.json().get("data", resp.json())

embedding_candidates = [m for m in models if m.get("custom_metadata", {}).get("model_type") == "embedding"]
if not embedding_candidates:
    raise RuntimeError(f"No embedding model found in /v1/models. Got: {models}")

embedding_model_id = embedding_candidates[0]["id"]
embedding_dimension = embedding_candidates[0]["custom_metadata"]["embedding_dimension"]
print(f"Embedding model: {embedding_model_id} ({embedding_dimension} dims)")

## Cell 9 — Create or find the vector store (idempotent)

In [ ]:
VECTOR_STORE_NAME = "uc02-complaint-intelligence"

list_resp = requests.get(f"{LLAMA_STACK_URL}/v1/vector_stores")
list_resp.raise_for_status()
existing = [s for s in list_resp.json().get("data", []) if s.get("name") == VECTOR_STORE_NAME]

if existing:
    VECTOR_STORE_ID = existing[0]["id"]
    print(f"Using existing store: {VECTOR_STORE_ID}")
else:
    create_resp = requests.post(
        f"{LLAMA_STACK_URL}/v1/vector_stores",
        json={
            "name": VECTOR_STORE_NAME,
            "embedding_model": embedding_model_id,
            "embedding_dimension": embedding_dimension,
        }
    )
    if not create_resp.ok:
        raise RuntimeError(f"Store creation failed ({create_resp.status_code}): {create_resp.text}")
    VECTOR_STORE_ID = create_resp.json()["id"]
    print(f"Created store: {VECTOR_STORE_ID}")

## Cell 10 — Upload-and-attach helper, plus existing-document lookup

`existing_ids_for_kind()` supports idempotent population: skip re-adding
anything already attached, so re-running this notebook is safe.

In [ ]:
def add_document(text: str, metadata: dict) -> str:
    """Uploads text as a file, attaches it to the vector store with metadata.
    Returns the file_id."""
    upload_resp = requests.post(
        f"{LLAMA_STACK_URL}/v1/files",
        files={"file": ("doc.txt", text.encode("utf-8"))},
        data={"purpose": "assistants"}
    )
    if not upload_resp.ok:
        raise RuntimeError(f"File upload failed ({upload_resp.status_code}): {upload_resp.text}")
    file_id = upload_resp.json()["id"]

    attach_resp = requests.post(
        f"{LLAMA_STACK_URL}/v1/vector_stores/{VECTOR_STORE_ID}/files",
        json={"file_id": file_id, "attributes": metadata}
    )
    if not attach_resp.ok:
        raise RuntimeError(f"Attach to store failed ({attach_resp.status_code}): {attach_resp.text}")
    return file_id


def existing_ids_for_kind(kind: str) -> set:
    """Returns the set of `id` attributes already attached for a given kind,
    so population loops can skip documents already present. Paginates using
    the `after` cursor (confirmed live 2026-07-22: this endpoint defaults to
    20 results per page and does not honor a `limit` override the way
    expected, that param is silently ignored rather than erroring)."""
    ids = set()
    after = None
    while True:
        params = {"after": after} if after else {}
        resp = requests.get(f"{LLAMA_STACK_URL}/v1/vector_stores/{VECTOR_STORE_ID}/files", params=params)
        resp.raise_for_status()
        page = resp.json()
        files = page.get("data", [])
        ids.update(
            f["attributes"]["id"] for f in files
            if f.get("attributes", {}).get("kind") == kind and "id" in f.get("attributes", {})
        )
        if not page.get("has_more") or not files:
            break
        after = files[-1]["id"]
    return ids


def search_by_kind(query: str, kind: str, top_k: int = 3) -> list:
    """Runs a filtered search, typed-filter syntax confirmed live 2026-07-22
    (a flat dict filter silently returns zero results rather than erroring,
    do not revert to that shape)."""
    resp = requests.post(
        f"{LLAMA_STACK_URL}/v1/vector_stores/{VECTOR_STORE_ID}/search",
        json={
            "query": query,
            "filters": {"type": "eq", "key": "kind", "value": kind},
            "max_num_results": top_k
        }
    )
    if not resp.ok:
        raise RuntimeError(f"Search failed ({resp.status_code}): {resp.text}")
    return resp.json().get("data", [])

print("Helpers defined: add_document, existing_ids_for_kind, search_by_kind")

## Cell 11 — Single-document round-trip checkpoint (validated 2026-07-22)

Confirms upload -> attach -> filtered search works end to end. Left in
place as a fast sanity check on future runs, not just a one-time gate.

In [ ]:
test_theme = taxonomy["themes"][0]
already_present = test_theme["id"] in existing_ids_for_kind("taxonomy")

if already_present:
    print(f"{test_theme['id']} already attached, skipping upload.")
else:
    test_text = f"{test_theme['name']}: {test_theme['definition'].strip()}"
    test_file_id = add_document(test_text, {"kind": "taxonomy", "item_type": "theme", "id": test_theme["id"]})
    print(f"Uploaded and attached: {test_file_id}")
    import time
    time.sleep(2)  # indexing may not be instant

results = search_by_kind(test_theme["definition"][:100], "taxonomy")
print(json.dumps(results, indent=2))
assert any(r["attributes"]["id"] == test_theme["id"] for r in results), \
    "Round-trip failed: expected document not found in filtered search results"
print("Round-trip confirmed: upload -> attach -> filtered search works.")

## Cell 12 — Populate the taxonomy (17 documents, idempotent)

One document per theme (10) and per root cause (7). Embeds `definition`,
`includes`, `excludes`, `examples` concatenated, matching taxonomy.yaml's
own stated intent that these fields are retrieval content, not decoration.

In [ ]:
already = existing_ids_for_kind("taxonomy")
added = 0

for t in taxonomy["themes"]:
    if t["id"] in already:
        continue
    parts = [f"{t['name']}: {t['definition'].strip()}"]
    if t.get("includes"):
        parts.append("Includes: " + "; ".join(t["includes"]))
    if t.get("excludes"):
        parts.append("Excludes: " + "; ".join(t["excludes"]))
    if t.get("examples"):
        parts.append("Examples: " + "; ".join(t["examples"]))
    add_document("\n".join(parts), {"kind": "taxonomy", "item_type": "theme", "id": t["id"]})
    added += 1

for r in taxonomy["root_causes"]:
    if r["id"] in already:
        continue
    text = f"{r['name']}: {r['definition'].strip()}"
    add_document(text, {"kind": "taxonomy", "item_type": "root_cause", "id": r["id"]})
    added += 1

print(f"Added {added} new taxonomy documents ({len(already)} already present).")

import time
if added:
    time.sleep(3)  # let indexing settle before the next cell counts

final_count = len(existing_ids_for_kind("taxonomy"))
expected = len(taxonomy["themes"]) + len(taxonomy["root_causes"])
print(f"Taxonomy documents in store: {final_count} (expected {expected})")
assert final_count == expected, "Taxonomy population incomplete, check for upload errors above."

## Cell 13 — Populate complaints (200 documents, idempotent)

Reads the full `records.jsonl` from MinIO, not just the first line. This
is the scope decision confirmed 2026-07-22: full corpus now, ahead of the
batch seeding job, rather than stubbing at one record.

In [ ]:
response = minio_client.get_object(COMPLAINTS_BUCKET, COMPLAINTS_KEY)
all_lines = response.read().decode("utf-8").splitlines()
response.close()
response.release_conn()

all_complaints = [json.loads(line) for line in all_lines]
print(f"Loaded {len(all_complaints)} complaint records from MinIO.")

already = existing_ids_for_kind("complaint")
added = 0

for c in all_complaints:
    if c["complaint_id"] in already:
        continue
    add_document(c["body"], {
        "kind": "complaint",
        "id": c["complaint_id"],
        "channel": c.get("channel", ""),
        "received_date": c.get("received_date", "")
    })
    added += 1
    if added % 25 == 0:
        print(f"  {added} uploaded...")

print(f"Added {added} new complaint documents ({len(already)} already present).")

import time
if added:
    time.sleep(5)

final_count = len(existing_ids_for_kind("complaint"))
print(f"Complaint documents in store: {final_count} (expected {len(all_complaints)})")
assert final_count == len(all_complaints), "Complaint population incomplete, check for upload errors above."

## Cell 14 — Retrieve context for the current complaint

Two filtered searches, combined into labeled sections. Excludes the
current complaint from its own retrieved context (a complaint should not
retrieve itself as a "similar past complaint").

In [ ]:
taxonomy_hits = search_by_kind(complaint["body"], "taxonomy", top_k=3)
complaint_hits = [
    r for r in search_by_kind(complaint["body"], "complaint", top_k=4)
    if r["attributes"].get("id") != complaint["complaint_id"]
][:3]

taxonomy_section = "\n".join(
    f"- {r['content'][0]['text']}" for r in taxonomy_hits
) if taxonomy_hits else "(none retrieved)"

complaint_section = "\n".join(
    f"- {r['content'][0]['text'][:200]}" for r in complaint_hits
) if complaint_hits else "(none retrieved)"

retrieved_context = (
    f"Relevant taxonomy guidance:\n{taxonomy_section}\n\n"
    f"Similar past complaints:\n{complaint_section}"
)

print(retrieved_context)

## Cell 15 — Build the classification prompt

Now wired to retrieval, `retrieved_context` comes from Cell 14 rather than
staying empty.

In [ ]:
theme_block = "\n".join(
    f"- {t['id']}: {t['name']} — {t['definition'].strip()}"
    for t in taxonomy["themes"]
)
root_cause_block = "\n".join(
    f"- {r['id']}: {r['name']} — {r['definition'].strip()}"
    for r in taxonomy["root_causes"]
)

prompt = f"""You are classifying a bank complaint against a fixed taxonomy.

Themes:
{theme_block}

Root causes:
{root_cause_block}

{retrieved_context}

Complaint:
\"\"\"{complaint['body']}\"\"\"

Respond with JSON only, no other text, in this exact shape:
{{
  "theme_id": "THM-XX",
  "root_cause_id": "RC-XX",
  "confidence": 0.0,
  "citation_text": "the exact sentence from the complaint that supports this classification",
  "candidate_themes": [{{"theme_id": "THM-XX", "score": 0.0}}, ...]
}}

candidate_themes must include every theme you considered plausible, ranked
by score, even if only one. citation_text must be copied verbatim from the
complaint text above."""

print(prompt[:800], "...")

## Cell 16 — Call the model and parse the response

In [ ]:
completion_resp = requests.post(
    f"{LLAMA_STACK_URL}/v1/chat/completions",
    json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 500,
        "temperature": 0.0
    }
)
completion_resp.raise_for_status()
raw = completion_resp.json()["choices"][0]["message"]["content"]

try:
    classification = json.loads(raw)
except json.JSONDecodeError:
    raise RuntimeError(f"Model did not return valid JSON:\n{raw}")

print(json.dumps(classification, indent=2))

## Cell 17 — Locate the citation and apply the review threshold

In [ ]:
citation_text = classification["citation_text"]
start = complaint["body"].find(citation_text)

if start == -1:
    citation = {"start": None, "end": None, "text": citation_text}
    citation_verified = False
else:
    citation = {"start": start, "end": start + len(citation_text), "text": citation_text}
    citation_verified = True

confidence = classification["confidence"]
routed_to_review = confidence < CONFIDENCE_THRESHOLD

if routed_to_review:
    sorted_candidates = sorted(classification["candidate_themes"],
                                key=lambda c: c["score"], reverse=True)
    review_reason = (
        f"confidence {confidence:.2f} below threshold {CONFIDENCE_THRESHOLD}"
    )
    candidate_theme_ids = [c["theme_id"] for c in sorted_candidates]
else:
    review_reason = None
    candidate_theme_ids = []

print(f"Routed to review: {routed_to_review}")
print(f"Citation verified against source text: {citation_verified}")

## Cell 18 — Assemble and write the evidence record

In [ ]:
from datetime import datetime, timezone
import uuid
import io

evidence_record = {
    "complaint_id": complaint["complaint_id"],
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "theme_id": classification["theme_id"],
    "root_cause_id": classification["root_cause_id"],
    "confidence": confidence,
    "citation": citation,
    "routed_to_review": routed_to_review,
    "review_reason": review_reason,
    "candidate_themes": candidate_theme_ids,
    "pii_detected": pii_detected,
    "pii_redactions": pii_redactions,
    "injection_blocked": injection_blocked,
    "guardrail_policy_id": "regex",
    "prompt_version": "0.1.0",
    "model_version": MODEL_ID,
    "taxonomy_version": TAXONOMY_VERSION,
    "trace_id": completion_resp.json().get("id", str(uuid.uuid4()))
}

record_bytes = json.dumps(evidence_record).encode("utf-8")
record_key = f"{EVIDENCE_PREFIX}/{complaint['complaint_id']}.json"
minio_client.put_object(
    EVIDENCE_BUCKET, record_key,
    data=io.BytesIO(record_bytes),
    length=len(record_bytes),
    content_type="application/json"
)

print(f"Evidence record written to s3://{EVIDENCE_BUCKET}/{record_key}")
print(json.dumps(evidence_record, indent=2))